In [3]:
import json

# Đọc file JSONL
def read_jsonl(file_path):
    """Đọc file JSONL và trả về list các dict"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Đọc kết quả từ file continuous
file_path = 'results/fire_vifactcheck_1.7b_continuous.jsonl'
raw_data = read_jsonl(file_path)
raw_data = raw_data[13:]  

print(f"Đã đọc {len(raw_data)} dòng từ {file_path}")
print(f"\nCác key trong mỗi record: {list(raw_data[0].keys())}")
print(f"\nVí dụ claim đầu tiên: {raw_data[0]['claim'][:100]}...")
print(f"Label: {raw_data[0]['label']}")
print(f"Result: {raw_data[0]['result']['answer']}")
print(f"Runtime: {raw_data[0]['runtime_seconds']:.2f}s")

Đã đọc 787 dòng từ results/fire_vifactcheck_1.7b_continuous.jsonl

Các key trong mỗi record: ['claim', 'label', 'result', 'searches', 'runtime_seconds']

Ví dụ claim đầu tiên: Đối tượng cố gắng bỏ chạy sau khi lực lượng công an đón dừng xe để kiểm tra....
Label: True
Result: Not Enough Info
Runtime: 132.02s


In [7]:
raw_data[0]

{'claim': 'Đối tượng cố gắng bỏ chạy sau khi lực lượng công an đón dừng xe để kiểm tra.',
 'label': 'True',
 'result': {'response': '{"final_answer": "Not Enough Info"}  \n**Giải thích:** Câu hỏi liên quan đến việc đối tượng cố gắng bỏ chạy khi cảnh sát kiểm tra, nhưng KN không có thông tin cụ thể về các tình huống này. Các nguồn kiến thức chỉ đề cập đến các biện pháp xử lý vi phạm và luật pháp liên quan, không đề cập đến các sự kiện cụ thể như việc đối tượng bỏ chạy. Do đó, không đủ thông tin để xác định tính đúng đắn của câu声明.',
  'answer': 'Not Enough Info'},
 'searches': {'google_searches': [{'query': 'police stop vehicle inspection 2026',
    'result': 'From 2026, traffic police will only handle violations if image or video evidence is available, a move to enhance transparency in law enforcement. Police forces are urged to crack down on major causes of road accidents, particularly driving under the influence of alcohol, speeding, overloading, improper ... Vietnam enforces a stric

In [8]:
import json
from sklearn.metrics import classification_report, precision_recall_fscore_support

# Dữ liệu đầu vào từ log của bạn
# raw_data = [
#     {"claim": "Chính phủ Nhật...", "label": "False", "result": {"answer": "Not Enough Info"}, "runtime_seconds": 103.3129},
#     {"claim": "Dự án đường cao tốc...", "label": "False", "result": {"answer": "True"}, "runtime_seconds": 82.3441}
# ]

def calculate_metrics(data):
    y_true = []
    y_pred = []
    total_runtime = 0

    for entry in data:
        y_true.append(entry['label'])
        y_pred.append(entry['result']['answer'])
        total_runtime += entry['runtime_seconds']

    # Tính toán chi tiết bằng sklearn
    # Zero_division=0 để tránh lỗi khi có nhãn không được dự đoán
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    
    print("--- KẾT QUẢ ĐÁNH GIÁ MODEL ---")
    print(f"Tổng số mẫu: {len(data)}")
    # print(f"Tổng thời gian chạy: {total_runtime:.2f} giây")
    print(f"Thời gian trung bình/mẫu: {total_runtime/len(data):.2f} giây")
    print("-" * 30)
    
    # In bảng kết quả
    print(f"{'Nhãn':<20} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
    for label, metrics in report.items():
        if label in ['accuracy', 'macro avg', 'weighted avg']:
            continue
        print(f"{label:<20} | {metrics['precision']:<10.2f} | {metrics['recall']:<10.2f} | {metrics['f1-score']:<10.2f}")
    
    print("-" * 30)
    print(f"Accuracy tổng thể: {report['accuracy']:.2f}")
    print(f"Precision trung bình: {precision:.2f}")
    print(f"Recall trung bình: {recall:.2f}")
    print(f"F1-Score trung bình: {f1:.2f}")
    

# if __name__ == "__main__":
calculate_metrics(raw_data)

--- KẾT QUẢ ĐÁNH GIÁ MODEL ---
Tổng số mẫu: 787
Thời gian trung bình/mẫu: 85.23 giây
------------------------------
Nhãn                 | Precision  | Recall     | F1-Score  
False                | 0.40       | 0.21       | 0.27      
Not Enough Info      | 0.32       | 0.69       | 0.44      
True                 | 0.42       | 0.16       | 0.23      
------------------------------
Accuracy tổng thể: 0.35
Precision trung bình: 0.38
Recall trung bình: 0.35
F1-Score trung bình: 0.31


In [ ]:
import polars as pl
from pathlib import Path
import os

In [ ]:
root_path = Path("G:\hcmus\khaithacdulieu\vifactcheck\data")
root_path
df = pl.read_parquet(f"{root_path}\train.parquet")